In [1]:
!git config --global user.email "beglossy@yonsei.ac.kr"
!git config --global user.name "beglossy-cmyk"

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
%cd gpt2.0_by_InaYoon
print("완료!")

Cloning into 'gpt2.0_by_InaYoon'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 14 (delta 4), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 9.43 KiB | 9.43 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/gpt2.0_by_InaYoon
완료!


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# 셰익스피어 데이터 다운로드
if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print(f"텍스트 길이: {len(text)}")
print(f"vocab_size: {vocab_size}")
print(f"예시: {text[:50]}")

텍스트 길이: 1115394
vocab_size: 65
예시: First Citizen:
Before we proceed any further, hear


In [3]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 3
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print(f"xb.shape: {xb.shape}")  # (64, 3)
print(f"yb.shape: {yb.shape}")  # (64, 3) ← 이번엔 y도 시퀀스!

xb.shape: torch.Size([64, 3])
yb.shape: torch.Size([64, 3])


In [4]:
class MLPCharacterModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=10, hidden_dim=200):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, emb_dim),
            nn.Flatten(),
            nn.Linear(block_size * emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        return self.net(x)

def sequence_cross_entropy(logits, targets):
    # logits: (B, T, V) → (B, V, T) 로 바꿔야 PyTorch loss 계산 가능
    return F.cross_entropy(logits.transpose(1, 2), targets)

model = MLPCharacterModel(vocab_size, block_size)

# 테스트
logits = model(xb)
print(f"logits.shape: {logits.shape}")  # (64, 27) ← 마지막 위치만 예측

logits.shape: torch.Size([64, 65])


In [5]:
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        # yb의 마지막 글자만 정답으로 사용
        loss = F.cross_entropy(logits, yb[:, -1])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 device: {device}")

model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for epoch in range(30):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    if epoch % 5 == 0 or epoch == 29:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

사용 device: cpu
epoch  0 | train loss 2.7510
epoch  5 | train loss 2.2463
epoch 10 | train loss 2.1606
epoch 15 | train loss 2.1113
epoch 20 | train loss 2.1049
epoch 25 | train loss 2.0848
epoch 29 | train loss 2.0972


In [6]:
@torch.no_grad()
def sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300))

ROMEO:
CARISCASTINIUT:
The, home my would mill hearder:
I fret her he do bet,
DHAMDIO:
Way what tufost heck?
DICINIUS:
And her ford wair whends and he willled spe, mardlikenossme.

QUTO:
The this noll on this net yeathe word enk?

FUCIO:
For!
Blot?
Take CopI
The coot of lank,
Hom, not, her,
He will le pre


In [ ]:
from google.colab import userdata, _message
import json

nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)
with open('/content/gpt2.0_by_InaYoon/notebook_03_mlp_shakespeare.ipynb', 'w') as f:
    json.dump(nb['ipynb'], f)

token = userdata.get('GITHUB_TOKEN')
%cd /content/gpt2.0_by_InaYoon
!git remote set-url origin https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
!git add notebook_03_mlp_shakespeare.ipynb
!git commit -m "Add notebook_03: MLP on Shakespeare"
!git push origin main